# 项目：天气识别--优化三

优化方式三：

1.分析错误原因从而做出针对性调整

###  训练策略演变
第一阶段v4：使用 Adam + ReduceOnPlateau 或标准余弦退火，达到约 90% 准确率。

第二阶段（针对 sunny‑cloudy 混淆）：

增强数据增强：提高 ColorJitter 参数至 brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1。

使用 Focal Loss（gamma=2.0）代替交叉熵，让模型更关注难分样本（如 sunny 与 cloudy 边界）。

继续微调：加载之前训练好的最佳模型（best_weather_model.pdparams），采用 更小学习率（0.0005）、余弦退火 + 预热（warmup 2 epochs），重新训练 15 个 epoch，并在验证循环中输出每个类别的分类报告。

# 一、分析错误原因的前置工作（修改val_dataset的类使之有图片信息并进行训练）

# 第一部分：数据集与v4模型准备

In [ ]:
# v5目的：实现可以查看错误
# 修改 val_dataset 的构建方式，使其能返回图片路径。修改原来的 TransformSubset 类，让它返回原始图片（未转换）或同时返回路径
class TransformSubsetWithPath(Dataset):
    def __init__(self, subset, transform, full_dataset):
        self.subset = subset
        self.transform = transform
        self.full_dataset = full_dataset   # 原始数据集（包含图片路径）

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]      # img 是 PIL Image
        # 得到原始图片在原始数据集中的索引
        original_idx = self.subset.indices[idx]
        img_path = self.full_dataset.samples[original_idx][0]
        if self.transform:
            img = self.transform(img)
        return img, label, img_path

In [ ]:
import os
import paddle
import paddle.vision.transforms as T
from paddle.io import Dataset, DataLoader, random_split
from PIL import Image

# 数据集根目录（请根据你的实际路径修改）
data_dir = 'data/weather/train'

# ==================== 1. 定义数据预处理 ====================
# train_transform = T.Compose([
#     T.Resize((224, 224)),
#     T.RandomHorizontalFlip(prob=0.5),          # 注意：Paddle 中用 prob
#     T.RandomRotation(15),                  # 旋转 ±15 度————保留一个这个！
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# 针对 sunny 和 cloudy 加强数据增强
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(prob=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),  # 增加饱和度、色调抖动
    # 原因：晴天和阴天的核心区别是对比度和颜色饱和度。增强这些属性可以让模型更关注光影和色彩差异
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==================== 2. 自定义数据集类 ====================
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()
        self.root_dir = root_dir
        self.transform = transform
        
        # 获取所有子文件夹名称并排序
        self.classes = sorted([d for d in os.listdir(root_dir) 
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        # 定义支持的图片后缀（可自行扩展）
        self.valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
        
        # 收集图片路径及标签
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            class_idx = self.class_to_idx[class_name]
            for item_name in os.listdir(class_dir):
                item_path = os.path.join(class_dir, item_name)
                # 仅当文件是普通文件且后缀为图片格式时才加入
                if os.path.isfile(item_path) and item_name.lower().endswith(self.valid_extensions):
                    self.samples.append((item_path, class_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# ==================== 3. 实例化数据集并划分 ====================
# 加载完整数据集（不带 transform，因为训练/验证需要不同增强）
full_dataset = CustomImageDataset(data_dir)

# 打印类别信息
print("类别列表:", full_dataset.classes)
print("类别映射:", full_dataset.class_to_idx)

# 划分训练集和验证集（80% 训练，20% 验证）
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# 因为 random_split 返回的是 Subset 对象，我们需要包装一层以应用不同的 transform
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]   # subset 返回 (img, label)
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = TransformSubset(train_subset, train_transform)
#val_dataset = TransformSubset(val_subset, val_transform)
val_dataset = TransformSubsetWithPath(val_subset, val_transform, full_dataset)  # 修改此处以使用新的 TransformSubsetWithPath 类

# ==================== 4. 创建 DataLoader ====================
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"训练样本数: {len(train_dataset)}，验证样本数: {len(val_dataset)}")

# 第二部分：模型定义、训练与验证

In [ ]:
!!pip install scikit-learn -q

### 以下代码块是v4的第二次重新跑的结果，本文件中主要用于得到含有错误图片路径的结果！

In [ ]:
import paddle
import paddle.nn.functional as F
from paddle.vision.models import resnet50
from paddle.optimizer.lr import ReduceOnPlateau

# ==================== 1. 构建模型 ====================
model = resnet50(pretrained=True)

# 关键：直接使用 ResNet50 的标准输出维度 2048 来替换全连接层
model.fc = paddle.nn.Linear(2048, 5)

# 验证替换是否正确
print("当前 fc 层:", model.fc)
print("fc 权重形状:", model.fc.weight.shape)   # 应为 [5, 2048]

# ==================== 2. 定义损失函数和优化器 ====================
#criterion = paddle.nn.CrossEntropyLoss()  #损失函数没变，要保留它
#-------------------------修改v5 -------------------------------------------------------------
class FocalLoss(paddle.nn.Layer):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = paddle.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(gamma=2.0)


#optimizer = paddle.optimizer.Adam(learning_rate=0.001, parameters=model.parameters())

# 原方案（ReduceOnPlateau）
# scheduler = paddle.optimizer.lr.ReduceOnPlateau(learning_rate=0.001, factor=0.5, patience=3)

# v3-新方案：余弦退火（从 0.001 衰减到 0），周期为总 epochs 数
#-------------------------v3-结合学习率预热（Warmup）
from paddle.optimizer.lr import LinearWarmup

epochs = 15

cosine_scheduler = paddle.optimizer.lr.CosineAnnealingDecay(
    learning_rate=0.001,
    T_max=epochs - 3,      # 预热 3 个 epoch 后的余弦周期
    eta_min=1e-6
)

scheduler = LinearWarmup(
    learning_rate=cosine_scheduler,
    warmup_steps=3 * len(train_loader),  # 3个epoch的步数（total batches）
    start_lr=1e-5,
    end_lr=0.001
)

# v3-然后创建优化器时传入 scheduler
optimizer = paddle.optimizer.AdamW(
    learning_rate=scheduler,
    weight_decay=0.01,
    parameters=model.parameters()
)

# ==================== 3. 训练与验证循环 ====================
epochs = 15
best_val_acc = 0.0

for epoch in range(epochs):
    # ---------- 训练阶段 ----------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_id, (images, labels) in enumerate(train_loader):
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        
        # 统计准确率
        preds = paddle.argmax(outputs, axis=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.shape[0]
        train_loss += loss.item() * labels.shape[0]
        
        if batch_id % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_id}] Loss: {loss.item():.4f}")
    
    train_acc = train_correct / train_total
    avg_train_loss = train_loss / train_total
    
    # ---------- 验证阶段 ----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_true = []   # 新增：存储所有真实标签
    all_pred = []   # 新增：存储所有预测标签
    
    with paddle.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            preds = paddle.argmax(outputs, axis=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.shape[0]
            val_loss += loss.item() * labels.shape[0]

            # 收集标签
            all_true.extend(labels.numpy())
            all_pred.extend(preds.numpy())
    
    val_acc = val_correct / val_total
    avg_val_loss = val_loss / val_total
    
    # 打印 epoch 统计信息

    # 打印总体指标
    print(f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    # 打印每个类别的详细指标
    print("\nClassification Report (Val):")
    print(classification_report(all_true, all_pred, target_names=full_dataset.classes, digits=4))
    print("-" * 50)
    
    # v3-学习率调度——每个 epoch 结束时调用
    scheduler.step()
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        paddle.save(model.state_dict(), "best_weather_model.pdparams")
        print(f"*** 保存最佳模型，验证准确率: {val_acc:.4f} ***")

print(f"训练完成，最佳验证准确率: {best_val_acc:.4f}")

# 二、v5-收集错误样本并可视化

### 1.训练结束后，收集错误样本并可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
from PIL import Image

model.eval()
error_records = []
all_true = []
all_pred = []

with paddle.no_grad():
    for images, labels, paths in val_loader:
        outputs = model(images)
        probs = F.softmax(outputs, axis=1)
        preds = paddle.argmax(probs, axis=1)
        
        for i in range(len(labels)):
            true_label = labels[i].item()
            pred_label = preds[i].item()
            all_true.append(true_label)
            all_pred.append(pred_label)
            if pred_label != true_label:
                error_records.append({
                    'path': paths[i],
                    'true_label': full_dataset.classes[true_label],
                    'pred_label': full_dataset.classes[pred_label],
                    'confidence': probs[i][pred_label].item()
                })

print(f"总验证样本数: {len(val_dataset)}，错误样本数: {len(error_records)}")

### 2.显示错误样本图像

In [ ]:
# 显示错误样本图像（最多显示 16 张）
def show_errors(errors, num=16):
    num = min(num, len(errors))
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()
    for i in range(num):
        err = errors[i]
        img = Image.open(err['path']).convert('RGB')
        axes[i].imshow(img)
        axes[i].set_title(f"True: {err['true_label']}\nPred: {err['pred_label']}\nConf: {err['confidence']:.2f}")
        axes[i].axis('off')
    for j in range(num, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

show_errors(error_records)

### 3.绘制混淆矩阵

In [ ]:
cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(full_dataset.classes))
plt.xticks(tick_marks, full_dataset.classes, rotation=45)
plt.yticks(tick_marks, full_dataset.classes)

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### 4.保存错误识别图片信息

In [ ]:
import csv
# 保存所有错误样本信息到 CSV，方便人工检查
with open('error_samples.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['image_path', 'true_label', 'pred_label', 'confidence'])
    for err in error_records:
        writer.writerow([err['path'], err['true_label'], err['pred_label'], err['confidence']])

### 分析：
sunny与cloudy的双方混淆比较严重

# 三、调整loss、学习率继续训练

In [ ]:
import paddle
import paddle.nn.functional as F
from paddle.vision.models import resnet50
from paddle.optimizer.lr import LinearWarmup
from sklearn.metrics import classification_report   # 新增

# 1. 构建模型
model = resnet50(pretrained=True)
model.fc = paddle.nn.Linear(2048, 5)

# 2. 加载之前训练好的最佳权重
model.load_state_dict(paddle.load("best_weather_model.pdparams"))

# 3. 定义 Focal Loss
class FocalLoss(paddle.nn.Layer):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = paddle.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(gamma=2.0)

# 4. 学习率调度（继续使用余弦退火 + 预热，但降低初始学习率）
epochs = 10   # 微调用更少的 epoch
cosine_scheduler = paddle.optimizer.lr.CosineAnnealingDecay(
    learning_rate=0.0005,   # 降低学习率
    T_max=epochs - 2,
    eta_min=1e-6
)
scheduler = LinearWarmup(
    learning_rate=cosine_scheduler,
    warmup_steps=2 * len(train_loader),
    start_lr=1e-5,
    end_lr=0.0005
)
optimizer = paddle.optimizer.AdamW(learning_rate=scheduler, weight_decay=0.01, parameters=model.parameters())

# 5. 训练循环（同上，略，注意 val_loader 返回两个值）
# ==================== 训练与验证循环 ====================
epochs = 15
best_val_acc = 0.0

for epoch in range(epochs):
    # ---------- 训练阶段 ----------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_id, (images, labels) in enumerate(train_loader):
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        
        # 统计准确率
        preds = paddle.argmax(outputs, axis=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.shape[0]
        train_loss += loss.item() * labels.shape[0]
        
        if batch_id % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_id}] Loss: {loss.item():.4f}")
    
    train_acc = train_correct / train_total
    avg_train_loss = train_loss / train_total
    
    # ---------- 验证阶段 ----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_true = []   # 新增：存储所有真实标签
    all_pred = []   # 新增：存储所有预测标签
    
    with paddle.no_grad():
        for images, labels, _  in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            preds = paddle.argmax(outputs, axis=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.shape[0]
            val_loss += loss.item() * labels.shape[0]

            # 收集标签
            all_true.extend(labels.numpy())
            all_pred.extend(preds.numpy())
    
    val_acc = val_correct / val_total
    avg_val_loss = val_loss / val_total
    
    # 打印 epoch 统计信息

    # 打印总体指标
    print(f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    # 打印每个类别的详细指标
    print("\nClassification Report (Val):")
    print(classification_report(all_true, all_pred, target_names=full_dataset.classes, digits=4))
    print("-" * 50)
    
    # v3-学习率调度——每个 epoch 结束时调用
    scheduler.step()
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        paddle.save(model.state_dict(), "best_weather_model.pdparams")
        print(f"*** 保存最佳模型，验证准确率: {val_acc:.4f} ***")

print(f"训练完成，最佳验证准确率: {best_val_acc:.4f}")

### 截断操作：
模型在第一个 epoch 后验证准确率达到 96.29%，但后续 epoch 验证准确率持续下降到 94-95%。这明显是过拟合或学习率过大导致训练震荡。此时，不应该继续训练下去，而应停止训练并恢复到最佳模型（epoch 1 的权重）

### 加载最佳模型并验证（第一轮训练的结果）

In [ ]:
# 加载最佳模型
model.load_state_dict(paddle.load("best_weather_model.pdparams"))
model.eval()

# 重新计算一次验证集指标（确保与训练时的验证集一致）
val_correct = 0
val_total = 0
with paddle.no_grad():
    for images, labels, _ in val_loader:
        outputs = model(images)
        preds = paddle.argmax(outputs, axis=1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.shape[0]
print(f"Final validation accuracy: {val_correct/val_total:.4f}")

### 训练结果观察
第 1 个 epoch：验证准确率飙升到 96.29%，分类报告显示 sunny 召回率 0.9635，cloudy 召回率 0.9624，两者混淆显著降低。

后续 epoch：验证准确率逐渐下降（95% → 94.5%），foggy 和 cloudy 的召回率出现波动，典型过拟合症状。

### 结论：最佳模型为第 1 个 epoch 的权重，后续训练反而损害泛化能力。

# 第三部分：推理示例（可选）
以下代码对单张图片进行预测。

In [ ]:
import numpy as np
from PIL import Image

def predict_image(image_path, model, class_names, transform):
    """对单张图片进行分类预测"""
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0)   # 增加 batch 维度
    
    with paddle.no_grad():
        output = model(image_tensor)
        prob = F.softmax(output, axis=1)
        pred_idx = paddle.argmax(prob, axis=1).item()
    
    return class_names[pred_idx], prob[0][pred_idx].item()

# 使用示例（请替换为你的图片路径）
test_image_path = 'data/weather/evaluation/cloudy/cloudy1.jpg'
pred_class, confidence = predict_image(test_image_path, model, full_dataset.classes, val_transform)
print(f"预测结果: {pred_class}，置信度: {confidence:.4f}")

test_image_path = 'data/weather/train/sunny/10281712.jpg'
pred_class, confidence = predict_image(test_image_path, model, full_dataset.classes, val_transform)
print(f"预测结果: {pred_class}，置信度: {confidence:.4f}")

test_image_path = 'data/weather/train/snowy/0833.jpg'
pred_class, confidence = predict_image(test_image_path, model, full_dataset.classes, val_transform)
print(f"预测结果: {pred_class}，置信度: {confidence:.4f}")


# 第四部分：适用其他数据集验证模型的泛化能力

In [ ]:
import os
import paddle
import paddle.nn.functional as F
from paddle.vision.models import resnet50
from paddle.vision.transforms import Compose, Resize, ToTensor, Normalize
from paddle.io import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# ==================== 配置 ====================
# 新数据集根目录（请修改为你的实际路径）
new_data_root = "data/weather/evaluation"  # 包含 cloudy, foggy, rainy, snowy, sunny 子文件夹

# 原数据集类别顺序（必须与模型输出索引一致）
original_classes = ['cloudy', 'foggy', 'rainy', 'snowy', 'sunny']

# 如果新数据集的文件夹名称与 original_classes 不完全一致，提供映射字典
# 例如:新数据集文件夹名为 'sun', 'cl', 'fg', 'rn', 'sn' 等
# 这里假设名称一致，若不一致请修改下面的映射
folder_to_label = {cls: idx for idx, cls in enumerate(original_classes)}
# 例如: folder_to_label = {'cloud': 0, 'fog': 1, 'rain': 2, 'snow': 3, 'sun': 4}

# ==================== 自定义数据集（仅用于测试，不进行数据增强）====================
class TestImageDataset(Dataset):
    def __init__(self, root_dir, folder_to_label, transform=None):
        self.transform = transform
        self.samples = []
        for folder_name, label in folder_to_label.items():
            folder_path = os.path.join(root_dir, folder_name)
            if not os.path.isdir(folder_path):
                print(f"警告：文件夹 {folder_name} 不存在于 {root_dir}")
                continue
            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                    img_path = os.path.join(folder_path, img_name)
                    self.samples.append((img_path, label))
        print(f"加载了 {len(self.samples)} 张测试图片")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label, img_path  # 返回路径便于分析错误样本

# ==================== 数据预处理（与训练时的验证预处理一致）====================
test_transform = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==================== 创建 DataLoader ====================
test_dataset = TestImageDataset(new_data_root, folder_to_label, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# ==================== 构建并加载模型 ====================
model = resnet50(pretrained=False)  # 不加载预训练权重，我们加载自己训练的
model.fc = paddle.nn.Linear(2048, 5)
model.load_state_dict(paddle.load("best_weather_model.pdparams"))  # 请确保路径正确
model.eval()

# ==================== 测试循环 ====================
all_true = []
all_pred = []
error_samples = []  # 存储错误样本的路径、真实标签、预测标签

with paddle.no_grad():
    for images, labels, paths in test_loader:
        outputs = model(images)
        probs = F.softmax(outputs, axis=1)
        preds = paddle.argmax(probs, axis=1)
        all_true.extend(labels.numpy())
        all_pred.extend(preds.numpy())
        for i in range(len(labels)):
            if preds[i] != labels[i]:
                error_samples.append({
                    'path': paths[i],
                    'true_label': original_classes[labels[i].item()],
                    'pred_label': original_classes[preds[i].item()],
                    'confidence': probs[i][preds[i]].item()
                })

# ==================== 评估指标 ====================
accuracy = sum(np.array(all_true) == np.array(all_pred)) / len(all_true)
print(f"测试集总样本数: {len(all_true)}")
print(f"测试准确率: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\n分类报告:")
print(classification_report(all_true, all_pred, target_names=original_classes, digits=4))

# 可选：混淆矩阵
cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix on New Dataset')
plt.colorbar()
tick_marks = np.arange(len(original_classes))
plt.xticks(tick_marks, original_classes, rotation=45)
plt.yticks(tick_marks, original_classes)
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# 可选：保存错误样本列表
with open('error_samples_new_dataset.txt', 'w') as f:
    for err in error_samples:
        f.write(f"{err['path']}\t{err['true_label']}\t{err['pred_label']}\t{err['confidence']:.4f}\n")
print(f"错误样本已保存到 error_samples_new_dataset.txt，共 {len(error_samples)} 个")

In [ ]:
import os
import paddle
import paddle.nn.functional as F
from paddle.vision.models import resnet50
from paddle.vision.transforms import Compose, Resize, ToTensor, Normalize
from paddle.io import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# ==================== 配置 ====================
# 新数据集根目录（请修改为你的实际路径）
new_data_root = "data/weather/evaluation"  # 包含 cloudy, foggy, rainy, snowy, sunny 子文件夹

# 原数据集类别顺序（必须与模型输出索引一致）
original_classes = ['Cloudy', 'Foggy', 'Rainy', 'Snowy', 'Sunny']

# 如果新数据集的文件夹名称与 original_classes 不完全一致，提供映射字典
# 例如:新数据集文件夹名为 'sun', 'cl', 'fg', 'rn', 'sn' 等
# 这里假设名称一致，若不一致请修改下面的映射
folder_to_label = {cls: idx for idx, cls in enumerate(original_classes)}
# 例如: folder_to_label = {'cloud': 0, 'fog': 1, 'rain': 2, 'snow': 3, 'sun': 4}

# ==================== 自定义数据集（仅用于测试，不进行数据增强）====================
class TestImageDataset(Dataset):
    def __init__(self, root_dir, folder_to_label, transform=None):
        self.transform = transform
        self.samples = []
        for folder_name, label in folder_to_label.items():
            folder_path = os.path.join(root_dir, folder_name)
            if not os.path.isdir(folder_path):
                print(f"警告：文件夹 {folder_name} 不存在于 {root_dir}")
                continue
            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                    img_path = os.path.join(folder_path, img_name)
                    self.samples.append((img_path, label))
        print(f"加载了 {len(self.samples)} 张测试图片")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label, img_path  # 返回路径便于分析错误样本

# ==================== 数据预处理（与训练时的验证预处理一致）====================
test_transform = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==================== 创建 DataLoader ====================
test_dataset = TestImageDataset(new_data_root, folder_to_label, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# ==================== 构建并加载模型 ====================
model = resnet50(pretrained=False)  # 不加载预训练权重，我们加载自己训练的
model.fc = paddle.nn.Linear(2048, 5)
model.load_state_dict(paddle.load("best_weather_model.pdparams"))  # 请确保路径正确
model.eval()

# ==================== 测试循环 ====================
all_true = []
all_pred = []
error_samples = []  # 存储错误样本的路径、真实标签、预测标签

with paddle.no_grad():
    for images, labels, paths in test_loader:
        outputs = model(images)
        probs = F.softmax(outputs, axis=1)
        preds = paddle.argmax(probs, axis=1)
        all_true.extend(labels.numpy())
        all_pred.extend(preds.numpy())
        for i in range(len(labels)):
            if preds[i] != labels[i]:
                error_samples.append({
                    'path': paths[i],
                    'true_label': original_classes[labels[i].item()],
                    'pred_label': original_classes[preds[i].item()],
                    'confidence': probs[i][preds[i]].item()
                })

# ==================== 评估指标 ====================
accuracy = sum(np.array(all_true) == np.array(all_pred)) / len(all_true)
print(f"测试集总样本数: {len(all_true)}")
print(f"测试准确率: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\n分类报告:")
print(classification_report(all_true, all_pred, target_names=original_classes, digits=4))

# 可选：混淆矩阵
cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix on New Dataset')
plt.colorbar()
tick_marks = np.arange(len(original_classes))
plt.xticks(tick_marks, original_classes, rotation=45)
plt.yticks(tick_marks, original_classes)
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# 可选：保存错误样本列表
with open('error_samples_new_dataset.txt', 'w') as f:
    for err in error_samples:
        f.write(f"{err['path']}\t{err['true_label']}\t{err['pred_label']}\t{err['confidence']:.4f}\n")
print(f"错误样本已保存到 error_samples_new_dataset.txt，共 {len(error_samples)} 个")

### 分析：泛化能力还是不够，尤其在预测cloudy的时候有问题